<a href="https://colab.research.google.com/github/lilian662/SIMULACION-I/blob/main/Lineadeesperacon_un_servidor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reporte de Simulación: Sistema de Líneas de Espera M/M/1
##Edith Lilian Castillo Ojeda
**Algoritmo:** Basado en el pseudocódigo de eventos discretos de *Sheldon Ross*.  
**Caso de Estudio:** $\lambda = 4$ (tasa de llegadas) y $\mu = 6$ (tasa de servicio).



## 1. Fundamento Teórico (Solución Analítica)
Para un sistema de una sola cola con un servidor ($M/M/1$), las soluciones teóricas de largo plazo se rigen por las siguientes fórmulas:

* **Factor de utilización ($\rho$):** Porcentaje de tiempo que el servidor está ocupado trabajando.
$$\rho = \frac{\lambda}{\mu} = \frac{4}{6} = 0.6667$$

* **Probabilidad de sistema vacío ($P_0$):** Porcentaje de tiempo que no hay nadie en el sistema.
$$P_0 = 1 - \rho = 1 - 0.6667 = 0.3333$$

* **Número promedio de unidades en cola ($L_q$):**
$$L_q = \frac{\lambda^2}{\mu(\mu - \lambda)} = \frac{4^2}{6(6 - 4)} = \frac{16}{12} = 1.3333 \text{ clientes}$$

* **Número promedio de unidades en el sistema ($L_s$):**
$$L_s = L_q + \frac{\lambda}{\mu} = 1.3333 + 0.6667 = 2.0000 \text{ clientes}$$

* **Tiempo promedio de espera en cola ($W_q$):**
$$W_q = \frac{L_q}{\lambda} = \frac{1.3333}{4} = 0.3333 \text{ unidades de tiempo}$$



# Algoritmo de Eventos Discretos (Sheldon Ross)

El algoritmo de Sheldon Ross simula el comportamiento de una línea de espera con un servidor mediante el avance del tiempo basado en **eventos futuros**, en lugar de avanzar el reloj segundo a segundo.

El sistema se rige por un reloj global ($t$), el estado actual del sistema ($n$, que representa el número de clientes adentro), y dos cronómetros clave que predicen el futuro:
* $t_A$: El momento exacto en el que llegará el **siguiente** cliente.
* $t_D$: El momento exacto en el que saldrá el cliente que está siendo atendido actualmente.


## Estructura de Control: Los 4 Casos de Ross

El bucle principal evalúa constantemente qué evento ocurrirá primero en el tiempo (`t_A` o `t_D`) y si este ocurre dentro del horario de atención permitido ($T$).

### 1. Caso 1: Próximo evento es una Llegada ($t_A \le t_D$) y ocurre a tiempo ($t_A \le T$)
* **¿Qué pasa?** El reloj del sistema avanza directamente hasta el momento de la llegada ($t = t_A$).
* **Acciones:** * Se registra el cliente agregándolo al contador ($N_A$) y al estado del sistema ($n = n + 1$).
    * Se programa de forma inmediata la **siguiente** llegada generando un tiempo aleatorio exponencial $T_r$, actualizando $t_A = t + T_r$.
    * Si el sistema estaba vacío ($n = 1$), este nuevo cliente pasa directo al servidor, por lo que se genera su tiempo de servicio $Y$ y se calcula su hora de salida: $t_D = t + Y$.

### 2. Caso 2: Próximo evento es una Salida ($t_D < t_A$) y ocurre a tiempo ($t_D \le T$)
* **¿Qué pasa?** El servidor termina de atender a alguien antes de que llegue un nuevo cliente. El reloj avanza a la hora de salida ($t = t_D$).
* **Acciones:**
    * El cliente sale del sistema, reduciendo el estado ($n = n - 1$) y sumando al contador de salidas ($N_D$).
    * Si el sistema queda completamente vacío ($n = 0$), el cronómetro de salidas se "apaga" colocándolo en el infinito ($t_D = \infty$).
    * Si todavía quedan clientes esperando en la cola ($n > 0$), el siguiente de la fila pasa al servidor. Se genera su tiempo de servicio $Y$ y se actualiza $t_D = t + Y$.

### 3. Caso 3: Se acabó el tiempo de arribos ($t > T$) pero aún quedan clientes en el sistema ($n > 0$)
* **¿Qué pasa?** El reloj superó el tiempo límite $T$, lo que significa que **ya no se permiten nuevas llegadas** ($t_A$ deja de importar). Sin embargo, el servidor no puede irse a casa porque quedan clientes adentro.
* **Acciones:**
    * El sistema ignora las llegadas y se enfoca únicamente en procesar las salidas pendientes ($t = t_D$).
    * Cada vez que un cliente sale, se reduce $n$ y se incrementa $N_D$.
    * Mientras $n > 0$, se sigue calculando el próximo $t_D = t + Y$ para limpiar la cola.

### 4. Caso 4: Se acabó el tiempo ($t > T$) y el sistema está completamente vacío ($n = 0$)
* **¿Qué pasa?** Es la condición de parada del algoritmo. Ya se rebasó el tiempo límite $T$ y se terminó de atender hasta el último cliente rezagado.
* **Acciones:**
    * Se calcula el tiempo extra remanente que el servidor tuvo que trabajar después de la hora de corte: $T_p = t - T$.
    * El bucle se rompe (`break`) y la simulación finaliza de forma exitosa, guardando los históricos de tiempos de llegada $A(i)$ y salida $D(i)$ para el análisis estadístico.

In [1]:
import random
import math

# 1. EL SIMULADOR PURO DE SHELDON ROSS (UNA SOLA CORRIDA)
def simulador_sheldon_ross(T, lam, mu):
    t, N_A, N_D, n = 0, 0, 0, 0
    t_A = -math.log(random.random()) / lam
    t_D = float('inf')
    A, D, historial = {}, {}, [(n, t)]

    while True:
    #Caso 1
        if t_A <= t_D and t_A <= T:
            t = t_A
            N_A += 1
            n += 1
            t_A = t + (-math.log(random.random()) / lam)
            if n == 1: t_D = t + (-math.log(random.random()) / mu)
            A[N_A] = t
            historial.append((n, t))
        #Caso 2
        elif t_D < t_A and t_D <= T:
            t = t_D
            n -= 1
            N_D += 1
            t_D = float('inf') if n == 0 else t + (-math.log(random.random()) / mu)
            D[N_D] = t
            historial.append((n, t))
        #Caso 3
        elif min(t_A, t_D) > T and n > 0:
            t = t_D
            n -= 1
            N_D += 1
            if n > 0: t_D = t + (-math.log(random.random()) / mu)
            D[N_D] = t
            historial.append((n, t))
        #Caso 4
        elif min(t_A, t_D) > T and n == 0:
            historial.append((n, t))
            break
    return N_A, N_D, A, D, historial

# --- EJECUCIÓN ÚNICA ---
lam, mu, T = 4.0, 6.0, 1000
N_A, N_D, A, D, historial = simulador_sheldon_ross(T, lam, mu)

# 2. EXTRACCIÓN DE MÉTRICAS SIMULADAS
t_final = historial[-1][1]
t_ocupado, t_vacio, t_cola, suma_clientes = 0, 0, 0, 0

for i in range(len(historial) - 1):
    n_act, t_act = historial[i]
    duracion = historial[i+1][1] - t_act
    suma_clientes += n_act * duracion
    if n_act == 0: t_vacio += duracion
    else: t_ocupado += duracion
    if n_act > 1: t_cola += (n_act - 1) * duracion

rho_sim = t_ocupado / t_final
P0_sim = t_vacio / t_final
Ls_sim = suma_clientes / t_final
Lq_sim = t_cola / t_final

tiempos_cola = [(A[i] if i == 1 else max(A[i], D[i-1])) - A[i] for i in range(1, N_D + 1)]
Wq_sim = sum(tiempos_cola) / len(tiempos_cola) if tiempos_cola else 0

# 3. VALORES TEÓRICOS (TUS FÓRMULAS)
rho_teo = lam / mu
P0_teo = 1 - rho_teo
Lq_teo = (lam**2) / (mu * (mu - lam))
Ls_teo = Lq_teo + rho_teo
Wq_teo = Lq_teo / lam
w=Wq_teo/mu



## 2. Validación y Análisis de Errores
El error porcentual se calcula utilizando la fórmula estándar:

$$\text{Error (\%)} = \left| \frac{\text{Valor Teórico} - \text{Valor Simulado}}{\text{Valor Teórico}} \right| \times 100$$

### Interpretación de los Resultados:
1. **Estado Transitorio vs. Estado Estable:** Al ejecutar la simulación una sola vez, los resultados reflejan un experimento aleatorio real. Por este motivo, el **Error %** nunca será exactamente $0.00\%$.
2. **Efecto del Azar:** Debido a la naturaleza estocástica del pseudocódigo de Sheldon Ross (el uso de `-math.log(random.random())`), habrá variaciones naturales en cada ejecución. Si el tiempo de corte $T$ es lo suficientemente grande (ej. $T = 1000$), los valores simulados convergerán y el error se mantendrá bajo, validando matemáticamente la precisión del algoritmo frente al modelo analítico.

In [2]:
# 4. IMPRESIÓN CON CÁLCULO DE ERROR DIRECTO
metricas = [
    ("Utilización (rho)", rho_teo, rho_sim),
    ("Sistema Vacío (P0)", P0_teo, P0_sim),
    ("Clientes en Cola (Lq)", Lq_teo, Lq_sim),
    ("Clientes en Sistema (Ls)", Ls_teo, Ls_sim),
    ("Tiempo en Cola (Wq)", Wq_teo, Wq_sim)


]

print(f"{'Métrica':<25} | {'Teórico':<8} | {'Simulado':<8} | {'Error':<8}")
print("-"*56)
for nombre, teo, sim in metricas:
    error_pct = (abs(teo - sim) / teo)
    print(f"{nombre:<25} | {teo:<8.4f} | {sim:<8.4f} | {error_pct:<7.2f}")

Métrica                   | Teórico  | Simulado | Error   
--------------------------------------------------------
Utilización (rho)         | 0.6667   | 0.6537   | 0.02   
Sistema Vacío (P0)        | 0.3333   | 0.3463   | 0.04   
Clientes en Cola (Lq)     | 1.3333   | 1.2278   | 0.08   
Clientes en Sistema (Ls)  | 2.0000   | 1.8815   | 0.06   
Tiempo en Cola (Wq)       | 0.3333   | 0.3071   | 0.08   
